In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from package.RankAMIP.logistic import run_logistic_regression
from package.RankAMIP.data_script import make_BT_design_matrix
from package.RankAMIP.logistic import LogisticAMIP
from package.RankAMIP.logistic import find_closest_matchups
from package.RankAMIP.logistic import isRankingRobust
from package.RankAMIP.data_script import *

### How Robust is the Multi-turn Benchmark to Data-Dropping?

The MT-Bench (Multi-Turn Benchmark) is a curated set of 80 multi-turn dialogue prompts designed to evaluate the conversational and instruction-following capabilities of large language models (LLMs). Each prompt simulates realistic, multi-turn interactions that test a model's ability to maintain context, reason logically, and provide coherent responses across various domains, including general knowledge, reasoning, programming, and open-ended tasks.

### Load Data

In [8]:
# Import datasets from 
# https://huggingface.co/datasets/lmsys/mt_bench_human_judgments
from datasets import load_dataset
ds = load_dataset("lmsys/mt_bench_human_judgments")

In [9]:
# inspect the available splits
print(ds)  
# MT-bench has both human and a gpt4-judge data.
gpt4_pair = ds["gpt4_pair"] 
human = ds["human"] 
# look at the first example
print(gpt4_pair[0])

DatasetDict({
    gpt4_pair: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 2400
    })
    human: Dataset({
        features: ['question_id', 'model_a', 'model_b', 'winner', 'judge', 'conversation_a', 'conversation_b', 'turn'],
        num_rows: 3355
    })
})
{'question_id': 81, 'model_a': 'alpaca-13b', 'model_b': 'claude-v1', 'winner': 'model_b', 'judge': 'gpt4_pair', 'conversation_a': [{'content': 'Compose an engaging travel blog post about a recent trip to Hawaii, highlighting cultural experiences and must-see attractions.', 'role': 'user'}, {'content': 'I recently had the pleasure of visiting Hawaii and it quickly became one of my favorite places. From the stunning beaches to the lush mountains, this place has it all. The people are incredibly friendly and the culture is alive and well. One of the highlights of my trip was visiting the Polynesian Cultural Center. Here, I was able 

In [10]:
df = gpt4_pair.to_pandas()
df.shape

(2400, 8)

In [11]:
# create a column winner_model_a, which is 1 if model_a is preferred, 0 if model_b is preferred
df['winner_model_a'] = df['winner'].apply(lambda x: 1 if x == 'model_a' else 0)
# create a column called winner_tie that is 1 if the winner is 'tie', else 0
df['winner_tie'] = df['winner'].apply(lambda x: 1 if x == 'tie' else 0)
df.head()

,question_id,model_a,model_b,winner,judge,conversation_a,conversation_b,turn,winner_model_a,winner_tie
0,81,alpaca-13b,claude-v1,model_b,gpt4_pair,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0
1,81,alpaca-13b,claude-v1,model_b,gpt4_pair,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2,0,0
2,81,alpaca-13b,gpt-3.5-turbo,model_b,gpt4_pair,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0
3,81,alpaca-13b,gpt-3.5-turbo,model_b,gpt4_pair,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,2,0,0
4,81,alpaca-13b,gpt-4,model_b,gpt4_pair,[{'content': 'Compose an engaging travel blog ...,[{'content': 'Compose an engaging travel blog ...,1,0,0


In [12]:
ties = df[df['winner_tie'] == 1]
print(f"Number of ties: {len(ties)}")
# proportion of ties.
print(f"Proportion of ties: {len(ties) / len(df):.2%}")
# note, the proportion of ties is 9.17% for the LLM-as-judge data and 23.25% for the human-as-judge data.

Number of ties: 220
Proportion of ties: 9.17%


In [ ]:
rawBT = df[['model_a', 'model_b', 'winner_model_a', 'winner_tie']] # ties dropped there were 2180 rows.
rawBT.head()
rawBT.shape

(2400, 4)

In [15]:
# how to get the unique names in both columns
model_a_names = df['model_a'].unique()
model_b_names = df['model_b'].unique()
# combine the two arrays and get the unique names
model_names = np.unique(np.concatenate((model_a_names, model_b_names)))
# print the number of unique model names
print(f"Number of unique model names: {len(model_names)}")

Number of unique model names: 6


In [18]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

alpaca-13b: 800
claude-v1: 800
gpt-3.5-turbo: 800
gpt-4: 800
llama-13b: 800
vicuna-13b-v1.2: 800


In [19]:
# make the BT design matrix.
X, y, player_to_id = make_BT_design_matrix(rawBT, weight_tie = True)
X.shape, y.shape

((4800, 5), (4800,))

#### Run Top-k Robustness Check.

In [20]:
ks = [1, 3, 5]

results = {}
for k in ks:
    alphaN = 1
    chatbotA = -1
    while chatbotA == -1:
        chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices = isRankingRobust(k, alphaN, X, y, weighted = True)
        results[(k, alphaN)] = (chatbotA, chatbotB, chatbotOriginalBetaDiff, chatNewBetaDiff, chatIndices)
        alphaN += 1

In [21]:
# find the (k, alpha N) pairs that are non-robust.
results_nonrobust = {k: v for k, v in results.items() if v[0] != -1}
results_nonrobust

{(1, 40): (1,
  4,
  0.3206212886452122,
  -0.008669570557585438,
  array([ 646,  587, 1290, 1741,  720,  570,  571,   72,  223, 1212, 1183,
         1122, 2052, 2053, 2112, 1242, 1063, 1033, 1032, 1003, 1812, 2113,
         1002,  282, 1093, 1092, 1243, 2022, 1753, 1752,  132,  103,  102,
         1872, 1873, 1543,  162, 1453, 1423, 1422])),
 (3, 158): (0,
  3,
  1.1289044762931844,
  -0.00017937436828430187,
  array([1703, 1522, 1631, 1751,  430,  131, 1721, 1120,  880, 1720,  580,
         1747,  217,  757,  216,  576,  726, 1746, 1537, 1656, 1477,  396,
           67,  429,  579, 1029,  548, 1718, 1299, 1689,  578,  158, 1028,
          428, 1238,  639,  638, 1118, 1298, 1749, 1748, 1538,  279, 2048,
          819, 1449, 1797, 1196, 1796, 1197, 1857,  446,  506, 2007, 2006,
         1046, 1977, 1976, 1047, 1947, 1946, 1076,  146,  147,  476, 1917,
         1887, 1886, 1136, 1226, 1856, 1167,  447,  176,  177, 1826, 1227,
         1467, 1767, 1376, 1377,  267, 1617, 1406, 1587, 1586

In [ ]:
# with open('results/MTBenchLLMNonrobustIF.pkl', 'wb') as f:
#     pickle.dump(results_nonrobust, f)

In [23]:
for model in model_names:
    filtered = df[
        (df['model_a'] == model) | 
        (df['model_b'] == model)
    ]
    print(f"{model}: {filtered.shape[0]}")

alpaca-13b: 800
claude-v1: 800
gpt-3.5-turbo: 800
gpt-4: 800
llama-13b: 800
vicuna-13b-v1.2: 800


In [24]:
from package.RankAMIP.plot_util import *
rankings = return_rankings_list(X, y, results, 1, 40, player_to_id)

In [25]:
# plot the rankings on the original arena
filename_to_save = 'fig/top6_mtb_llm.png'
plot_title = 'Model Rankings in MT-Bench'
plot_bt_scores(X, y, rankings, alphaN, 6, plot_title, filename_to_save)